# 05. Build the Artemis II SDS archive with `flovopy.ingest`, then cut event waveforms

The raw downloads from the **20260403_service** run are staged instrument by instrument with
`flovopy.ingest`, then merged into the master SDS archive (`/Volumes/KSCGTdownld/SDS`, from
`2026_ingest_service_runs/config/ksc.yaml`). Each instrument is staged into its own
`20_archive/<instrument>/SDS` folder first, and the staging is merged into the master as a separate,
transactional step (`flovopy.sds.merge_sds_archives`, can be rolled back).

**The master SDS has already been built.** All ingest cells therefore run with `DRY_RUN = True`:
they read headers, check the serial→SEED mappings and report counts, but write nothing.
Set `DRY_RUN = False` only to rebuild.

Section 8 (event waveform files) always writes: it cuts the launch window from the master SDS.

Paths, launch time and channel groups are in `artemis2_config.py` (shared with notebooks 25 and 35).

## 1. Configuration

In [ ]:
from pprint import pprint
from pathlib import Path

import artemis2_config as cfg
from flovopy.ingest.service import ingest_sds
from flovopy.ingest.gem import stage_gem_mseed
from flovopy.ingest.siliconaudio import convert_siliconaudio
from flovopy.ingest.smartsolo import stage_smartsolo_mseed
from flovopy.ingest.miniseed import stage_miniseed_tree
from flovopy.sds.merge_sds_archives import discover_sds_files

DRY_RUN = True   # the master SDS is already built; False re-stages and merges

RUN_DIR = cfg.RUN_DIR
DOWNLOAD = RUN_DIR / '00_download'
STAGING = {name: RUN_DIR / '20_archive' / name / 'SDS'
           for name in ('Gem', 'SiliconAudio', 'SmartSolo', 'Guralp')}

print('project  :', cfg.PROJECT_YAML)
print('run      :', RUN_DIR)
print('master   :', cfg.MASTER_SDS)
print('dry run  :', DRY_RUN)
print('launch   :', cfg.LAUNCHTIME, ' window', cfg.STARTTIME, '-', cfg.ENDTIME)

In [ ]:
def show(result, max_errors=10):
    """Compact summary of a flovopy.ingest result dictionary."""
    if not isinstance(result, dict):
        print(result); return
    for key, value in result.items():
        if key in ('errors', 'written', 'paths', 'rows'):
            continue
        print(f'{key:>14}: {value}')
    errors = result.get('errors') or []
    print(f'{"errors":>14}: {len(errors)}')
    for e in errors[:max_errors]:
        print('   ', e)
    if len(errors) > max_errors:
        print(f'    ... {len(errors) - max_errors} more')

## 2. USF Nanometrics Centaur stations (Trillium Compact, infraBSU, Chaparral)

Centaurs record SDS natively, so their SDS tree is merged straight into the master with
`ingest_sds` (fast mode: non-colliding files are copied byte for byte; files already in the master
are read and compared). In this run `00_download/Centaur` is a symlink to `20_archive/`, the Centaur
SDS recovered from the cards (about 3,700 day files).

Because nearly every file already exists in the master SDS, even the dry run reads and compares each
one, which takes roughly 9 hours on the USB drive. It is therefore off by default
(`CHECK_CENTAUR = False`).

`00_download/Centaur0/` holds full-SEED (`.seed`) exports from the B20 Centaur; flovopy.ingest has no
adapter for full SEED, and they are kept only as provenance.

In [ ]:
CHECK_CENTAUR = False   # ~9 h even as a dry run (compares every file with the master)
CENTAUR_SDS_SOURCE = DOWNLOAD / 'Centaur'   # symlink to ../20_archive in this run

if not CHECK_CENTAUR:
    print('Centaur check skipped (CHECK_CENTAUR = False).')
elif not CENTAUR_SDS_SOURCE.is_dir():
    print(f'No Centaur SDS at {CENTAUR_SDS_SOURCE}')
else:
    n_files = sum(1 for _ in discover_sds_files(CENTAUR_SDS_SOURCE))
    print(f'{n_files} Centaur SDS files in {CENTAUR_SDS_SOURCE.resolve()}')
    show(ingest_sds(cfg.PROJECT_YAML, CENTAUR_SDS_SOURCE, mode='fast', dry_run=DRY_RUN))

## 3. Boise State Gem infrasound loggers

The microSD cards were copied to `00_download/Gem/raw/` and decoded with gemlog's `gemconvert`
(hourly MiniSEED in `10_gemconvert/mseed/`). `stage_gem_mseed` renames each trace from the Gem serial
to its SEED ID using the campaign table `gem_mapping_artemis2_example.csv`.

Gem **363** maps to the placeholder `1R.UNK..HD1`. Strict staging rejects its files until the
station is identified, so they show up below as errors.

In [ ]:
GEM_MSEED = RUN_DIR / '10_gemconvert' / 'mseed'
gem = stage_gem_mseed(GEM_MSEED, STAGING['Gem'], mapping=cfg.GEM_MAPPING_CSV,
                      dry_run=DRY_RUN, strict=True, verbose=False)
show(gem, max_errors=5)

## 4. Silicon Audio (B23)

The Gecko SD card was recovered to `00_download/SiliconAudio/B23/` (`data/` and `histogram/`).
`convert_siliconaudio` merges the 1-minute MiniSEED files into day files.

In [ ]:
SA_DATA = DOWNLOAD / 'SiliconAudio' / 'B23' / 'data'
sa = convert_siliconaudio(SA_DATA, STAGING['SiliconAudio'], year=cfg.LAUNCHTIME.year,
                          dry_run=DRY_RUN, verbose=False)
show(sa)

## 5. SmartSolo geophones

DLD files are harvested with SoloLite and exported as one MiniSEED file per channel per day
(`serial.day.YYYY.MM.DD.HH.MM.SS.000.<E|N|Z>.miniseed`). `stage_smartsolo_mseed` maps serials to
`1R.<STA>..DP` plus the E/N/Z component (`smartsolo_mapping_artemis2.csv`).

The Artemis II export is not in this service run's `00_download/SmartSolo/` on this drive; the
original was at `/Volumes/tachyon/SOLODATA/prospect_ksc/artemis2/ten_stations`.

In [ ]:
SMARTSOLO_SOURCE = DOWNLOAD / 'SmartSolo'
# SMARTSOLO_SOURCE = Path('/Volumes/tachyon/SOLODATA/prospect_ksc/artemis2/ten_stations')

if SMARTSOLO_SOURCE.is_dir() and any(SMARTSOLO_SOURCE.rglob('*.miniseed')):
    solo = stage_smartsolo_mseed(SMARTSOLO_SOURCE, STAGING['SmartSolo'],
                                 mapping_csv=cfg.SMARTSOLO_MAPPING_CSV, recursive=True,
                                 dry_run=DRY_RUN, verbose=False)
    show(solo)
else:
    print(f'No SmartSolo MiniSEED in {SMARTSOLO_SOURCE}; skipped.')

## 6. Guralp stations (MSFC: B07, B12)

Downloaded through the Guralp web interface (`B07/`, `B12/`) and with wget (`B12/sd/`). The
MiniSEED headers already carry the `DG` network IDs, so the generic MiniSEED stager is used. Only
seismic streams are staged: the `S1Seis*` files and the web-interface `DG.*` files (the wget folder
also contains clock, power and state-of-health streams).

In [ ]:
GURALP = DOWNLOAD / 'Guralp'
guralp = stage_miniseed_tree(GURALP, STAGING['Guralp'], patterns=('DG.*.mseed', '*S1Seis*.mseed'),
                             dry_run=DRY_RUN, verbose=False)
show(guralp)

## 7. Merge staged instruments into the master SDS

Each staging archive is merged transactionally (`ingest_sds`); a merge session can be rolled back
with `flovopy-ingest --config ksc.yaml rollback <transaction_id>`. In a dry run nothing was staged,
so only staging folders that already exist are checked.

In [ ]:
for name, stage in STAGING.items():
    if not stage.is_dir():
        print(f'{name}: no staging archive at {stage}')
        continue
    print(f'== {name}')
    show(ingest_sds(cfg.PROJECT_YAML, stage, mode='fast', dry_run=DRY_RUN))

## 8. Launch-window waveforms from the master SDS

Check what the master SDS holds for the launch window, then write one Seisan WAV file per
instrument group (`cfg.CHANNEL_GROUPS`) to `cfg.EVENT_WAV_DIR`. Existing WAV files are not overwritten.

In [ ]:
from flovopy.seisanio.utils.helpers import write_wavfile

WRITE_EVENT_WAVS = True
client = cfg.sds_client()
streams = {}
for group in cfg.CHANNEL_GROUPS:
    st = cfg.get_group_stream(group, client)
    streams[group] = st
    ids = sorted({tr.id for tr in st})
    print(f'{group:>9}: {len(ids):3d} channels, {len(st):3d} traces  {ids}')

In [ ]:
if cfg.SKIPPED_FILES:
    # An incomplete WAV would be kept by skip_existing=True, so refuse to write any.
    print('NOT writing event WAVs: unreadable SDS files were skipped:')
    for path, err in cfg.SKIPPED_FILES:
        print('   ', path, '|', err)
elif WRITE_EVENT_WAVS:
    for group, st in streams.items():
        if len(st):
            print(write_wavfile(st, cfg.EVENT_WAV_DIR, group, len(st)))

In [ ]:
streams['SmartSolo'].plot(equal_scale=True, starttime=cfg.STARTTRIM, endtime=cfg.ENDTRIM);

## Appendix. SmartSolo vertical-channel StationXML (one-off)

Builds `KSC_SmartSolo_verticals.xml` from a SmartSolo template response and the station table, then
test-corrects the SmartSolo verticals. Paths point at `/Volumes/tachyon` and `~/Downloads`, so this is
off by default (`BUILD_SMARTSOLO_XML = False`).

In [ ]:
BUILD_SMARTSOLO_XML = False

if BUILD_SMARTSOLO_XML:
    from copy import deepcopy
    from pathlib import Path

    import pandas as pd
    from obspy import UTCDateTime, read_inventory
    from obspy.core.inventory import Inventory, Network, Station, Channel, Site, Equipment

    csv_file = Path("~/Downloads/for_chatgpt.csv")
    template_xml = Path("/Volumes/tachyon/FDSN_Information_453013851_1.xml")
    out_xml = Path("/Volumes/tachyon/KSC_SmartSolo_verticals.xml")

    # Read the SmartSolo template response
    template_inv = read_inventory(template_xml)
    template_chan = template_inv[0][0][0]
    smartsolo_response = template_chan.response

    # Use timing / sample-rate defaults from the template XML
    start_date = template_chan.start_date
    end_date = template_chan.end_date
    start_date = UTCDateTime(2026,3,10)
    end_date=UTCDateTime(2026,4,5)
    sample_rate = template_chan.sample_rate
    location_code = template_chan.location_code or "SW"
    location_code = ''
    channel_code = template_chan.code  # DPZ

    # Read station table
    df = pd.read_csv(csv_file, encoding="utf-8-sig")

    # Keep only rows with SmartSolo serial numbers
    df = df.dropna(subset=["SmartSolo serial number"]).copy()
    df["serial"] = df["SmartSolo serial number"].astype(int).astype(str)

    net = Network(
        code="1R",
        description="KSC SmartSolo vertical geophone network",
        start_date=start_date,
    )

    for _, row in df.iterrows():
        station_code = str(row["Station"]).strip()
        serial = row["serial"]

        lat = float(row["Lat (o)"])
        lon = float(row["Lon (o)"])

        # If you have real elevations later, replace this.
        elev = 0.0

        sta = Station(
            code=station_code,
            latitude=lat,
            longitude=lon,
            elevation=elev,
            site=Site(name=f"{station_code} SmartSolo {serial}"),
            start_date=start_date,
            end_date=end_date,
        )

        chan = Channel(
            code=channel_code,
            location_code=location_code,
            latitude=lat,
            longitude=lon,
            elevation=elev,
            depth=0.0,
            azimuth=0.0,
            dip=90.0,          # vertical, positive down in StationXML convention
            sample_rate=sample_rate,
            start_date=start_date,
            end_date=end_date,
        )

        chan.sensor = Equipment(
            type="DTSOLO",
            description="SmartSolo digital geophone",
            manufacturer="DTCC",
            serial_number=serial,
        )

        chan.response = deepcopy(smartsolo_response)

        sta.channels.append(chan)
        net.stations.append(sta)

    inv_out = Inventory(
        networks=[net],
        source="USF / generated from SmartSolo station table",
        sender="USF",
    )

    print(inv_out)
    inv_out.write(out_xml, format="STATIONXML")
    print(f"Wrote {out_xml}")

In [ ]:
if BUILD_SMARTSOLO_XML:
    st = streams['SmartSolo']
    st2 = st.copy()
    for tr in st2.select(component='Z'):
        print(tr)
        tr.remove_response(inv_out)
    #st2.remove_response(inv_out)
    st2.select(component='Z').plot(starttime=launchtime-30, endtime=launchtime+120);